# Neo4j Graph RAG

Complete flow: ingestion, graph construction, retrieval, grounded answers, tests, and Gradio.


In [ ]:
%pip install -q pandas neo4j python-dotenv gradio anthropic


In [1]:
from pathlib import Path
import os
import re

import pandas as pd
import gradio as gr
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "processed" / "property_listings_raw.csv"

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", os.getenv("NEO4J_USER", "neo4j"))
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

assert DATA_PATH.exists(), f"CSV not found: {DATA_PATH}"
assert NEO4J_URI and NEO4J_PASSWORD, "Set NEO4J_URI and NEO4J_PASSWORD in .env"

df = pd.read_csv(DATA_PATH)
print("Loaded rows:", len(df))
print("Columns:", df.columns.tolist())
df.head()


Loaded rows: 6495
Columns: ['id', 'crm_id', 'strapi_id', 'slug', 'title', 'listing_type', 'department', 'search_type', 'status', 'price', 'price_qualifier', 'beds', 'bathrooms', 'area', 'community', 'building', 'display_address', 'address.address1', 'address.address2', 'address.address3', 'address.address4', 'latitude', 'longitude', 'floorarea_min', 'floorarea_type', 'description', 'long_description', 'accommodation_summary', 'new_development', 'link', 'maps_url', 'source', 'images', 'image_count', 'scraped_at', 'introtext', 'special', 'brochure_url', 'dld_listing_number', 'dld_permit_type', 'dld_qr_url', 'updated_at', 'completion_status', 'cheques', 'plot_area_min', 'agent_name', 'agent_email', 'agent_phone', 'agent_whatsapp', 'agent_designation', 'agent_languages', 'agent_photo_url', 'agent_slug', 'video_tour', 'virtual_tour', 'enriched_at', 'property_type']


,id,crm_id,strapi_id,slug,title,listing_type,department,search_type,status,price,...,agent_phone,agent_whatsapp,agent_designation,agent_languages,agent_photo_url,agent_slug,video_tour,virtual_tour,enriched_at,property_type
0,9971,DBI215069,9971,2-bedroom-apartment-for-sale-in-mosela-watersi...,2 bedroom apartment for sale,purchase,residential,sales,For Sale,2850000.0,...,+971 58 577 0306,+971 58 577 0306,Greens & Views Consultant,"[""English"",""Turkish""]",https://ggfx-handh3.s3.eu-west-2.amazonaws.com...,murat-topcuoglu,[],[],2026-09-17T17:03:40Z,apartment
1,12292,DUB203155,12292,3-bedroom-villa-for-sale-in-district-8-jumeira...,3 bedroom villa for sale,purchase,residential,sales,For Sale,7195000.0,...,+971 50 598 0269,+971 50 598 0269,Associate Partner,"[""English""]",https://ggfx-handh3.s3.eu-west-2.amazonaws.com...,cathal-o-neill,[],[],2026-09-17T18:04:47Z,villa
2,10051,DBI210323,10051,2-bedroom-apartment-for-sale-in-al-ghozlan-1-t...,2 bedroom apartment for sale,purchase,residential,sales,For Sale,2250000.0,...,+971 58 577 0306,+971 58 577 0306,Greens & Views Consultant,"[""English"",""Turkish""]",https://ggfx-handh3.s3.eu-west-2.amazonaws.com...,murat-topcuoglu,[],[],2026-09-17T18:04:47Z,apartment
3,6452,JVG220178,6452,4-bedroom-villa-for-sale-in-bungalows-area-dub...,4 bedroom villa for sale,purchase,residential,sales,For Sale,9250000.0,...,+971 50 494 0897,+971 50 494 0897,Associate Director / Sales Team Manager - Vict...,"[""English""]",https://ggfx-handh3.s3.eu-west-2.amazonaws.com...,asim-sheikh,"[{""url"":""https://youtu.be/1H-h00b__Ng""}]","[{""url"":""https://youtu.be/1H-h00b__Ng""}]",2026-09-17T18:04:47Z,villa
4,5191,PAL230746,5191,1-bedroom-apartment-for-sale-in-al-sarrood-pal...,1 bedroom apartment for sale,purchase,residential,sales,For Sale,2750000.0,...,+971 58 526 3709,+971 58 526 3709,Palm Jumeirah Consultant,"[""English""]",https://ggfx-handh3.s3.eu-west-2.amazonaws.com...,ella-white,[],[],2026-09-17T18:04:47Z,apartment


In [2]:
def first_existing(row, *columns, default=None):
    for column in columns:
        if column in row.index and pd.notna(row[column]):
            return row[column]
    return default


def clean_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        value = value.strip()
        return value or None
    if hasattr(value, "item"):
        value = value.item()
    return value


def normalize_properties(frame):
    records = []

    for index, row in frame.iterrows():
        property_id = first_existing(
            row, "id", "property_id", "listing_id", "crm_id", default=index
        )

        record = {
            "property_id": str(property_id).strip(),
            "crm_id": clean_value(first_existing(row, "crm_id")),
            "title": clean_value(first_existing(row, "title", "name")),
            "description": clean_value(
                first_existing(row, "long_description", "description")
            ),
            "listing_type": clean_value(first_existing(row, "listing_type")),
            "status": clean_value(first_existing(row, "status")),
            "property_type": clean_value(
                first_existing(row, "property_type", "type")
            ),
            "price": clean_value(first_existing(row, "price")),
            "beds": clean_value(first_existing(row, "beds", "bedrooms")),
            "bathrooms": clean_value(
                first_existing(row, "bathrooms", "baths")
            ),
            "area": clean_value(
                first_existing(row, "area", "size", "area_sqft")
            ),
            "community": clean_value(
                first_existing(row, "community", "community_name")
            ),
            "building": clean_value(
                first_existing(row, "building", "building_name")
            ),
            "address": clean_value(
                first_existing(row, "display_address", "address")
            ),
            "latitude": clean_value(first_existing(row, "latitude", "lat")),
            "longitude": clean_value(first_existing(row, "longitude", "lon")),
            "url": clean_value(first_existing(row, "link", "url")),
        }

        records.append(record)

    properties = pd.DataFrame(records)
    properties = properties.drop_duplicates("property_id")
    properties = properties[
        properties["property_id"].notna()
        & properties["property_id"].ne("")
    ]
    return properties


properties = normalize_properties(df)
print("Unique properties:", len(properties))
properties.head()


Unique properties: 6495


,property_id,crm_id,title,description,listing_type,status,property_type,price,beds,bathrooms,area,community,building,address,latitude,longitude,url
0,9971,DBI215069,2 bedroom apartment for sale,haus & haus and Daniel are pleased to present ...,purchase,For Sale,apartment,2850000.0,2,2,Mosela,The Views,"[""Apartment""]","Mosela Waterside Residences, The Views, Dubai",25.088956,55.173342,https://www.hausandhaus.com/properties/for-sal...
1,12292,DUB203155,3 bedroom villa for sale,haus & haus is delighted to present this 3-bed...,purchase,For Sale,villa,7195000.0,3,4,District 8,Jumeirah Park,"[""Villa""]","District 8, Jumeirah Park, Dubai",25.038419,55.165733,https://www.hausandhaus.com/properties/for-sal...
2,10051,DBI210323,2 bedroom apartment for sale,haus & haus is proud to offer this two bedroom...,purchase,For Sale,apartment,2250000.0,2,2,Al Ghozlan,The Greens,"[""Apartment""]","Al Ghozlan 1, The Greens, Dubai",25.091872,55.173538,https://www.hausandhaus.com/properties/for-sal...
3,6452,JVG220178,4 bedroom villa for sale,haus & haus is thrilled to bring to the market...,purchase,For Sale,villa,9250000.0,4,5,Green Community,Dubai Investment Park,"[""Villa""]","Bungalows Area, Dubai Investment Park, Dubai",24.993891,55.166159,https://www.hausandhaus.com/properties/for-sal...
4,5191,PAL230746,1 bedroom apartment for sale,haus & haus is proud to present this 1 bedroom...,purchase,For Sale,apartment,2750000.0,1,2,Shoreline Apartments,Palm Jumeirah,"[""Apartment""]","Al Sarrood, Palm Jumeirah, Dubai",25.111924,55.142570,https://www.hausandhaus.com/properties/for-sal...


In [3]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
)

driver.verify_connectivity()
print("Connected to Neo4j")


Connected to Neo4j


In [4]:
def create_constraints():
    queries = [
        "CREATE CONSTRAINT property_id IF NOT EXISTS "
        "FOR (p:Property) REQUIRE p.property_id IS UNIQUE",
        "CREATE CONSTRAINT community_name IF NOT EXISTS "
        "FOR (c:Community) REQUIRE c.name IS UNIQUE",
        "CREATE CONSTRAINT building_name IF NOT EXISTS "
        "FOR (b:Building) REQUIRE b.name IS UNIQUE",
        "CREATE CONSTRAINT property_type_name IF NOT EXISTS "
        "FOR (t:PropertyType) REQUIRE t.name IS UNIQUE",
    ]

    with driver.session(database=NEO4J_DATABASE) as session:
        for query in queries:
            session.run(query).consume()


create_constraints()


In [5]:
def build_graph(properties, batch_size=500):
    rows = properties.to_dict(orient="records")

    query = '''
    UNWIND $rows AS row
    MERGE (p:Property {property_id: row.property_id})
    SET p += row

    FOREACH (_ IN CASE WHEN row.community IS NOT NULL
        THEN [1] ELSE [] END |
        MERGE (c:Community {name: row.community})
        MERGE (p)-[:LOCATED_IN]->(c)
    )

    FOREACH (_ IN CASE WHEN row.building IS NOT NULL
        THEN [1] ELSE [] END |
        MERGE (b:Building {name: row.building})
        MERGE (p)-[:IN_BUILDING]->(b)
    )

    FOREACH (_ IN CASE WHEN row.property_type IS NOT NULL
        THEN [1] ELSE [] END |
        MERGE (t:PropertyType {name: row.property_type})
        MERGE (p)-[:HAS_TYPE]->(t)
    )
    '''

    with driver.session(database=NEO4J_DATABASE) as session:
        for start in range(0, len(rows), batch_size):
            batch = rows[start:start + batch_size]
            session.run(query, rows=batch).consume()

    return len(rows)


ingested_count = build_graph(properties)
print("Properties processed:", ingested_count)


Properties processed: 6495


In [6]:
def run_query(query, parameters=None):
    with driver.session(database=NEO4J_DATABASE) as session:
        return [record.data() for record in session.run(query, parameters or {})]


def graph_counts():
    query = '''
    MATCH (p:Property)
    WITH count(p) AS properties
    OPTIONAL MATCH (c:Community)
    WITH properties, count(c) AS communities
    OPTIONAL MATCH (b:Building)
    WITH properties, communities, count(b) AS buildings
    OPTIONAL MATCH (t:PropertyType)
    RETURN properties, communities, buildings, count(t) AS property_types
    '''
    return pd.DataFrame(run_query(query))


def graph_sample(limit=5):
    return pd.DataFrame(run_query(
        '''
        MATCH (p:Property)
        RETURN p.property_id AS property_id,
               p.title AS title,
               p.price AS price,
               p.community AS community
        ORDER BY p.property_id
        LIMIT $limit
        ''',
        {"limit": limit},
    ))


display(graph_counts())
display(graph_sample())


,properties,communities,buildings,property_types
0,6495,233,15,27


,property_id,title,price,community
0,100112,1 bedroom apartment to rent,80000.0,Business Bay
1,100121,3 bedroom apartment to rent,225000.0,Dubai Hills Estate
2,100187,apartment to rent,50000.0,Jumeirah Village Circle
3,100236,2 bedroom apartment to rent,125000.0,Dubai Creek Harbour
4,100291,5 bedroom villa to rent,350000.0,Dubai Hills Estate


In [20]:

def graph_keyword_search(query, limit=8):
    terms = [
        term.lower()
        for term in re.findall(r"[a-zA-Z0-9]+", query)
        if len(term) > 1
    ]

    if not terms:
        return pd.DataFrame()

    # Detect whether the user wants to buy or rent
    transaction_filter = ""

    if re.search(r"\b(for sale|to buy|buy|purchase)\b", query, re.I):
        transaction_filter = """
        AND (
            toLower(coalesce(p.listing_type, '')) IN ['sale', 'purchase', 'buy']
            OR toLower(coalesce(p.status, '')) CONTAINS 'sale'
            OR toLower(coalesce(p.title, '')) CONTAINS 'for sale'
            OR toLower(coalesce(p.url, '')) CONTAINS '/for-sale/'
        )
        """

    elif re.search(r"\b(for rent|to rent|rent|rental|lease)\b", query, re.I):
        transaction_filter = """
        AND (
            toLower(coalesce(p.listing_type, '')) IN ['rent', 'rental', 'lease']
            OR toLower(coalesce(p.status, '')) CONTAINS 'rent'
            OR toLower(coalesce(p.title, '')) CONTAINS 'to rent'
            OR toLower(coalesce(p.url, '')) CONTAINS '/to-rent/'
        )
        """

    cypher = f"""
    MATCH (p:Property)
    OPTIONAL MATCH (p)-[:LOCATED_IN]->(c:Community)
    OPTIONAL MATCH (p)-[:IN_BUILDING]->(b:Building)
    OPTIONAL MATCH (p)-[:HAS_TYPE]->(t:PropertyType)

    WITH p, c, b, t,
         toLower(
             coalesce(p.title, '') + ' ' +
             coalesce(p.description, '') + ' ' +
             coalesce(p.community, '') + ' ' +
             coalesce(p.building, '') + ' ' +
             coalesce(p.property_type, '') + ' ' +
             coalesce(p.address, '')
         ) AS searchable

    WITH p, c, b, t, searchable,
         size([
             term IN $terms
             WHERE searchable CONTAINS term
         ]) AS score

    WHERE score > 0
    {transaction_filter}

    RETURN
        p.property_id AS property_id,
        p.title AS title,
        p.price AS price,
        p.beds AS beds,
        p.bathrooms AS bathrooms,
        p.area AS area,
        coalesce(c.name, p.community) AS community,
        coalesce(b.name, p.building) AS building,
        coalesce(t.name, p.property_type) AS property_type,
        p.listing_type AS listing_type,
        p.status AS status,
        p.address AS address,
        p.url AS url,
        score

    ORDER BY score DESC
    LIMIT $limit
    """

    return pd.DataFrame(
        run_query(cypher, {
            "terms": terms,
            "limit": int(limit)
        })
    )



def get_property(property_id):
    rows = run_query(
        '''
        MATCH (p:Property {property_id: $property_id})
        OPTIONAL MATCH (p)-[:LOCATED_IN]->(c:Community)
        OPTIONAL MATCH (p)-[:IN_BUILDING]->(b:Building)
        OPTIONAL MATCH (p)-[:HAS_TYPE]->(t:PropertyType)
        RETURN p.property_id AS property_id,
               p.title AS title,
               p.description AS description,
               p.price AS price,
               p.beds AS beds,
               p.bathrooms AS bathrooms,
               p.area AS area,
               coalesce(c.name, p.community) AS community,
               coalesce(b.name, p.building) AS building,
               coalesce(t.name, p.property_type) AS property_type,
               p.address AS address,
               p.url AS url
        ''',
        {"property_id": str(property_id).strip()},
    )
    return pd.DataFrame(rows)

def get_all_properties(limit=None):
    query = """
    MATCH (p:Property)
    OPTIONAL MATCH (p)-[:LOCATED_IN]->(c:Community)
    OPTIONAL MATCH (p)-[:IN_BUILDING]->(b:Building)
    OPTIONAL MATCH (p)-[:HAS_TYPE]->(t:PropertyType)
    RETURN
        p.property_id AS property_id,
        p.title AS title,
        p.price AS price,
        p.beds AS beds,
        p.bathrooms AS bathrooms,
        coalesce(t.name, p.property_type) AS property_type,
        p.area AS area,
        coalesce(c.name, p.community) AS community,
        p.address AS address,
        p.url AS url
    ORDER BY p.property_id
    """

    if limit is not None:
        query += " LIMIT $limit"
        rows = run_query(query, {"limit": int(limit)})
    else:
        rows = run_query(query)

    return pd.DataFrame(rows)

def search_community(name, limit=10):
    return pd.DataFrame(run_query(
        '''
        MATCH (p:Property)-[:LOCATED_IN]->(c:Community)
        WHERE toLower(c.name) CONTAINS toLower($name)
        RETURN p.property_id AS property_id,
               p.title AS title,
               p.price AS price,
               p.beds AS beds,
               p.area AS area,
               c.name AS community,
               p.url AS url
        ORDER BY p.price
        LIMIT $limit
        ''',
        {"name": name, "limit": int(limit)},
    ))


def community_overview(name):
    rows = run_query(
        '''
        MATCH (p:Property)-[:LOCATED_IN]->(c:Community)
        WHERE toLower(c.name) CONTAINS toLower($name)
        RETURN c.name AS community,
               count(p) AS property_count,
               avg(toFloat(p.price)) AS average_price,
               min(toFloat(p.price)) AS minimum_price,
               max(toFloat(p.price)) AS maximum_price
        ORDER BY property_count DESC
        LIMIT 1
        ''',
        {"name": name},
    )
    return pd.DataFrame(rows)


def portfolio_overview():
    return pd.DataFrame(run_query(
        '''
        MATCH (p:Property)
        RETURN count(p) AS property_count,
               count(DISTINCT p.community) AS communities,
               count(DISTINCT p.building) AS buildings,
               avg(toFloat(p.price)) AS average_price,
               min(toFloat(p.price)) AS minimum_price,
               max(toFloat(p.price)) AS maximum_price
        '''
    ))


print("Search methods ready")


Search methods ready


In [ ]:
def answer_question(question, limit=8):
    question = question.strip()

    if not question:
        return "Please enter a question.", pd.DataFrame()

    lower_question = question.lower()
    
    if re.search(
    r"\b(all|every|entire)\b.*\b(properties|property|listings|data)\b"
    r"|\b(show|display|list|get)\s+all\b",
    question,
    re.IGNORECASE,
    ):
        result = get_all_properties()
        return (
            f"Retrieved all {len(result)} properties from the Neo4j graph.",
            result,
        )
    
    if any(term in lower_question for term in [
        "portfolio overview", "total properties", "how many properties",
        "overall portfolio", "total listings"
    ]):
        result = portfolio_overview()
        if result.empty:
            return "No portfolio data found.", result
        row = result.iloc[0]
        answer = (
            f"The graph contains {int(row['property_count'])} properties "
            f"across {int(row['communities'])} communities and "
            f"{int(row['buildings'])} buildings. "
            f"Average listed price: {row['average_price']}. "
            f"Minimum price: {row['minimum_price']}; "
            f"maximum price: {row['maximum_price']}."
        )
        return answer, result

    community_match = re.search(
        r"(?:in|about|for)\s+(.+?)(?:\?|$)",
        question,
        re.IGNORECASE,
    )

    if "community" in lower_question and community_match:
        name = community_match.group(1).strip(" .?!")
        result = community_overview(name)
        if not result.empty:
            row = result.iloc[0]
            answer = (
                f"{row['community']} has {int(row['property_count'])} "
                f"properties in the graph. Average price: "
                f"{row['average_price']}; range: {row['minimum_price']} "
                f"to {row['maximum_price']}."
            )
            return answer, result

    id_match = re.search(
        r"(?:property\s*(?:id)?\s*[:#]?\s*)([A-Za-z0-9_-]+)",
        question,
        re.IGNORECASE,
    )

    if id_match:
        result = get_property(id_match.group(1))
        if not result.empty:
            return result.iloc[0].to_json(indent=2), result

    result = graph_keyword_search(question, limit=limit)

    if result.empty:
        return (
            "I couldn't find matching properties in the graph. "
            "Try a community, property type, building, or listing ID.",
            result,
        )

    lines = []
    for _, row in result.iterrows():
        lines.append(
            f"- ID: {row['property_id']} | "
            f"Title: {row.get('title')} | "
            f"Price: {row.get('price')} | "
            f"Community: {row.get('community')} | "
            f"Type: {row.get('property_type')}"
        )

    answer = (
        f"Found {len(result)} matching properties in the Neo4j graph:\n\n"
        + "\n".join(lines)
    )

    api_key = os.getenv("ANTHROPIC_API_KEY")
    if api_key:
        try:
            import anthropic

            client = anthropic.Anthropic(api_key=api_key)
            model = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-5")
            response = client.messages.create(
                model=model,
                max_tokens=700,
                system=(
                    "Answer using only the supplied property records. "
                    "If information is missing, say so. Do not invent "
                    "prices, features, or availability."
                ),
                messages=[{
                    "role": "user",
                    "content": (
                        f"Question: {question}\n\n"
                        f"Property evidence:\n{result.to_json(orient='records')}"
                    ),
                }],
            )
            answer = response.content[0].text
        except Exception as error:
            answer += f"\n\nLLM generation unavailable: {error}"

    return answer, result


sample_question = "apartments"
sample_answer, sample_evidence = answer_question(sample_question)
print(sample_answer)
display(sample_evidence.head())


# Apartments for Sale in Dubai

Based on the property records, here are the available apartments:

## 1-Bedroom Apartments

| Location | Area | Price | Bathrooms |
|----------|------|-------|-----------|
| Creek Horizon Tower 1, Dubai Creek Harbour | Creek Horizon | AED 1,800,000 | 1 |
| Address Residences Zabeel 3, Zabeel | Zabeel 1 | AED 2,200,000 | 1 |
| Sobha Orbis Tower A, Motor City | Sobha Orbis | AED 1,300,000 | 1 |
| Sobha Orbis Tower A, Motor City | Sobha Orbis | AED 1,250,000 | 1 |
| Eleve by Deyaar, Jebel Ali | Eleve By Deyaar | AED 1,150,000 | 1 |

## 2-Bedroom Apartments

| Location | Area | Price | Bathrooms |
|----------|------|-------|-----------|
| Ellison By Nshama Properties, Town Square | Ellison By Nshama Properties | AED 1,550,000 | 3 |
| The Highgrove By Ellington, Bukadra | The Highgrove By Ellington | AED 3,499,999 | 2 |
| Al Andalus Tower E, Jumeirah Golf Estates | Al Andalus | AED 2,500,000 | 2 |

**Price Range:** AED 1,150,000 to AED 3,499,999


,property_id,title,price,beds,bathrooms,area,community,building,property_type,address,url,score
0,105296,1 bedroom apartment for sale,1800000.0,1,1,Creek Horizon,Dubai Creek Harbour,"[""Apartment""]",apartment,"Creek Horizon Tower 1, Dubai Creek Harbour, Dubai",https://www.hausandhaus.com/properties/for-sal...,1
1,120153,2 bedroom apartment for sale,1550000.0,2,3,Ellison By Nshama Properties,Town Square,"[""Apartment""]",apartment,"Ellison By Nshama Properties, Town Square, Dubai",https://www.hausandhaus.com/properties/for-sal...,1
2,123930,1 bedroom apartment for sale,2200000.0,1,1,Zabeel 1,Zabeel,"[""Apartment""]",apartment,"Address Residences Zabeel 3, Zabeel, Dubai",https://www.hausandhaus.com/properties/for-sal...,1
3,139678,1 bedroom apartment for sale,1300000.0,1,1,Sobha Orbis,Motor City,"[""Apartment""]",apartment,"Sobha Orbis Tower A, Motor City, Dubai",https://www.hausandhaus.com/properties/for-sal...,1
4,139675,1 bedroom apartment for sale,1250000.0,1,1,Sobha Orbis,Motor City,"[""Apartment""]",apartment,"Sobha Orbis Tower A, Motor City, Dubai",https://www.hausandhaus.com/properties/for-sal...,1


In [9]:
def run_tests():
    counts = graph_counts().iloc[0]
    assert int(counts["properties"]) == len(properties), (
        f"Expected {len(properties)} properties, got {counts['properties']}"
    )

    assert int(counts["communities"]) > 0, "No community nodes found"

    sample_id = str(properties.iloc[0]["property_id"])
    assert not get_property(sample_id).empty, (
        f"Could not retrieve property {sample_id}"
    )

    search_results = graph_keyword_search("apartment", limit=5)
    assert isinstance(search_results, pd.DataFrame)

    answer, evidence = answer_question("apartments")
    assert isinstance(answer, str) and answer.strip()
    assert isinstance(evidence, pd.DataFrame)

    print("All end-to-end tests passed.")


run_tests()


All end-to-end tests passed.


In [13]:
def gradio_search(question):
    answer, evidence = answer_question(question)
    return answer, evidence


with gr.Blocks(title="Neo4j Property Graph RAG") as demo:
    gr.Markdown("# Property Graph RAG")
    gr.Markdown("Search property listings stored in Neo4j.")

    question_input = gr.Textbox(
        label="Ask about properties",
        placeholder="Find apartments in a community or enter a property ID",
        lines=2,
    )

    search_button = gr.Button("Search")
    answer_output = gr.Markdown()
    evidence_output = gr.Dataframe(
        label="Retrieved property evidence",
        interactive=False,
    )

    search_button.click(
        fn=gradio_search,
        inputs=question_input,
        outputs=[answer_output, evidence_output],
    )

    question_input.submit(
        fn=gradio_search,
        inputs=question_input,
        outputs=[answer_output, evidence_output],
    )

    gr.Examples(
        examples=[
            ["apartments"],
            ["villa"],
            ["portfolio overview"],
            ["community in Downtown"],
        ],
        inputs=question_input,
    )

demo.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [14]:
run_query("""
MATCH (p:Property)
RETURN
    toLower(coalesce(p.listing_type, 'unknown')) AS listing_type,
    count(p) AS total
ORDER BY total DESC
""")

[{'listing_type': 'rental', 'total': 4973},
 {'listing_type': 'purchase', 'total': 1522}]

In [15]:
run_query("""
MATCH (p:Property)
WHERE toLower(coalesce(p.title, '')) CONTAINS 'apartment'
RETURN
    p.property_id AS property_id,
    p.title AS title,
    p.listing_type AS listing_type,
    p.status AS status,
    p.price AS price,
    p.url AS url
LIMIT 30
""")

[{'property_id': '9971',
  'title': '2 bedroom apartment for sale',
  'listing_type': 'purchase',
  'status': 'For Sale',
  'price': 2850000.0,
  'url': 'https://www.hausandhaus.com/properties/for-sale/in-dubai/2-bedroom-apartment-for-sale-in-mosela-waterside-residences-the-views-dubai/9971'},
 {'property_id': '10051',
  'title': '2 bedroom apartment for sale',
  'listing_type': 'purchase',
  'status': 'For Sale',
  'price': 2250000.0,
  'url': 'https://www.hausandhaus.com/properties/for-sale/in-dubai/2-bedroom-apartment-for-sale-in-al-ghozlan-1-the-greens-dubai/10051'},
 {'property_id': '5191',
  'title': '1 bedroom apartment for sale',
  'listing_type': 'purchase',
  'status': 'For Sale',
  'price': 2750000.0,
  'url': 'https://www.hausandhaus.com/properties/for-sale/in-dubai/1-bedroom-apartment-for-sale-in-al-sarrood-palm-jumeirah-dubai/5191'},
 {'property_id': '298',
  'title': '2 bedroom apartment for sale',
  'listing_type': 'purchase',
  'status': 'For Sale',
  'price': 2550000.

In [16]:
run_query("""
MATCH (p:Property)
WHERE
    toLower(coalesce(p.title, '')) CONTAINS 'sale'
    OR toLower(coalesce(p.listing_type, '')) CONTAINS 'sale'
    OR toLower(coalesce(p.url, '')) CONTAINS 'for-sale'
RETURN
    p.property_id AS property_id,
    p.title AS title,
    p.listing_type AS listing_type,
    p.price AS price,
    p.url AS url
LIMIT 20
""")

[{'property_id': '9971',
  'title': '2 bedroom apartment for sale',
  'listing_type': 'purchase',
  'price': 2850000.0,
  'url': 'https://www.hausandhaus.com/properties/for-sale/in-dubai/2-bedroom-apartment-for-sale-in-mosela-waterside-residences-the-views-dubai/9971'},
 {'property_id': '12292',
  'title': '3 bedroom villa for sale',
  'listing_type': 'purchase',
  'price': 7195000.0,
  'url': 'https://www.hausandhaus.com/properties/for-sale/in-dubai/3-bedroom-villa-for-sale-in-district-8-jumeirah-park-dubai/12292'},
 {'property_id': '10051',
  'title': '2 bedroom apartment for sale',
  'listing_type': 'purchase',
  'price': 2250000.0,
  'url': 'https://www.hausandhaus.com/properties/for-sale/in-dubai/2-bedroom-apartment-for-sale-in-al-ghozlan-1-the-greens-dubai/10051'},
 {'property_id': '6452',
  'title': '4 bedroom villa for sale',
  'listing_type': 'purchase',
  'price': 9250000.0,
  'url': 'https://www.hausandhaus.com/properties/for-sale/in-dubai/4-bedroom-villa-for-sale-in-bungalo

In [18]:
run_query("""
MATCH (n:Property) 
WHERE n.property_id = '9971'
RETURN properties(n) AS complete_details

""")

[{'complete_details': {'area': 'Mosela',
   'address': 'Mosela Waterside Residences, The Views, Dubai',
   'latitude': 25.088956,
   'description': "haus & haus and Daniel are pleased to present this spacious 2 bedroom apartment in Mosela Waterside Residences, The Views. With a generous BUA of 1,416 sq.ft, this property comprises 2 bathrooms, a storage closet, an equipped kitchen and a large balcony providing epic lake views. Vacant and ready for immediate occupation.<br><br>The Views is perfectly located adjacent to Sheikh Zayed Road, this allows for easy access to some of Dubai's main attractions such as the Marina.<br><br>Please call for more information, to arrange a viewing or to make an offer.<br><br>Finance is available on this property through haus & haus partners. Please speak to us for more information.<br> <br>For further details, please drop into our flagship office at the Gold & Diamond Park – or browse the incredible selection of properties we maintain at the haus & haus 

In [ ]:
driver.close()
print("Neo4j driver closed")
